In [0]:
%sql
use catalog trueanalytics_data;

In [0]:
import pyspark
import pyspark.sql.functions as F
import pyspark.sql.types as T 
from functools import partial
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta

In [0]:
def save_to_parquet(df, save_path):
    (df.write.format('parquet')
        .mode('overwrite')
        .option("header", "true")
        .save(save_path)
    )
    return print("successfully save parquet to ", save_path)

def save_to_csv(df, save_path):
    (df.coalesce(1)
        .write.format('csv')
        .mode('overwrite')
        .option("header", "true")
        .save(save_path)
    )
    return print("successfully save csv to ", save_path)

In [0]:
# parameter: par_month
dbutils.widgets.text("par_month", "202510")
par_month = dbutils.widgets.get("par_month")

try:
  par_month = int(par_month)
except ValueError:
  par_month = 0
  raise ValueError("par_month value must be numeric")

if par_month!=0:
  pass
else:
  dbutils.notebook.exit("Aborting as ondition not met. Further tasks will be skipped")

# debug
display(par_month)

In [0]:
# master data
prep_path = f'dbfs:/Volumes/int-cu-siampiwat/staging/raw/{par_month}_footfall.parquet'
prep_freq_path = f'dbfs:/Volumes/int-cu-siampiwat/staging/raw/{par_month}_flag_freq.parquet'
prep_feature_360 = f'dbfs:/Volumes/int-cu-siampiwat/staging/raw/{par_month}_360_feature.parquet'
profile_path = 'dbfs:/Volumes/int-cu-siampiwat/staging/tmp/profile_chula.csv'
date_path = 'dbfs:/Volumes/int-cu-siampiwat/staging/tmp/day_type_apr_june_26.csv'
nantional_path = 'dbfs:/Volumes/int-cu-siampiwat/staging/tmp/country_group_chula.csv'
home_region_path = 'dbfs:/Volumes/int-cu-siampiwat/staging/tmp/home_region_chula.csv'

# report path
report_path = f'dbfs:/Volumes/int-cu-siampiwat/staging/report/report4/{par_month}/'

In [0]:
df_date = (spark.read
      .option("header", "true").option("inferSchema", "true")
      .csv(date_path)).select('date','WEEK','day_type_final')
    

In [0]:
df = spark.read.parquet(prep_path)
     # raw footfall by mall & customertype (is_bmr, is_non_bmr, is_foriegner)

df_freq = spark.read.parquet(prep_freq_path) # raw freq by mall
df_360 = spark.read.parquet(prep_feature_360)
     # raw profile
df_merge = df.join(F.broadcast(df_freq), ['msisdn','name'], 'left')\
    .join(F.broadcast(df_360.drop('is_bmr','is_non_bmr','is_foriegner','a_country_name','demo_tourist_sim_v1_tourist_bin','roaming_flag')), ['msisdn'], 'inner')
df_merge = df_merge\
    .join(df_date, [df_merge.par_day == df_date.date], "left")\
    .withColumnRenamed('par_month','month')
display(df_merge.count())

In [0]:
# Gender
df_intermediate_gender = df_merge\
    .withColumn('gender_female', F.when(F.col('gender') == 'F', F.lit(1)).otherwise(F.lit(0)))\
        .withColumn('gender_male', F.when(F.col('gender') == 'M', F.lit(1)).otherwise(F.lit(0)))\
            .withColumn('gender_unidentified', F.when(F.col('gender') == 'U', F.lit(1)).otherwise(F.lit(0)))

In [0]:
# Age_1_12
df_intermediate_age = df_intermediate_gender\
    .withColumn('age_1_12', F.when(F.col("age_range") == '1_12', F.lit(1)).otherwise(F.lit(0)))\
        .withColumn('age_13_17', F.when(F.col("age_range") == '13_17', F.lit(1)).otherwise(F.lit(0)))\
            .withColumn('age_18_24', F.when(F.col("age_range") == '18_24', F.lit(1)).otherwise(F.lit(0)))\
                .withColumn('age_25_34', F.when(F.col("age_range") == '25_34', F.lit(1)).otherwise(F.lit(0)))\
                    .withColumn('age_35_44', F.when(F.col("age_range") == '35_44', F.lit(1)).otherwise(F.lit(0)))\
                        .withColumn('age_45_54', F.when(F.col("age_range") == '45_54', F.lit(1)).otherwise(F.lit(0)))\
                            .withColumn('age_55_59', F.when(F.col("age_range") == '55_59', F.lit(1)).otherwise(F.lit(0)))\
                                .withColumn('age_60_64', F.when(F.col("age_range") == '60_64', F.lit(1)).otherwise(F.lit(0)))\
                                    .withColumn('age>=65', F.when(F.col("age_range") == '>=65', F.lit(1)).otherwise(F.lit(0)))\
                                    .withColumn('age_unidentified', F.when(F.col("age_range")=='unidentified', F.lit(1)).otherwise(F.lit(0)))

In [0]:
# Normal Weekday
df_intermediate_date = df_intermediate_age.withColumn('normal_weekday', F.when(F.col("day_type_final") == 'Normal Weekday', F.lit(1)).otherwise(F.lit(0)))
# Special Weekend
df_intermediate_date = df_intermediate_date.withColumn('special_weekend', F.when(F.col("day_type_final") == 'Special Weekend', F.lit(1)).otherwise(F.lit(0)))
# Normal Weekend
df_intermediate_date = df_intermediate_date.withColumn('normal_weekend', F.when(F.col("day_type_final") == 'Normal Weekend', F.lit(1)).otherwise(F.lit(0)))
# Holiday Weekend
df_intermediate_date = df_intermediate_date.withColumn('holiday_weekend', F.when(F.col("day_type_final") == 'Holiday Weekend', F.lit(1)).otherwise(F.lit(0)))
# Special Weekday
df_intermediate_date = df_intermediate_date.withColumn('special_weekday', F.when(F.col("day_type_final") == 'Special Weekday', F.lit(1)).otherwise(F.lit(0)))

# df_intermediate_date.display()

In [0]:
df_intermediate_hr = df_intermediate_date.withColumn('visitor_cnt_hour_10', F.when(F.col("par_hour") == 10, F.lit(1)).otherwise(F.lit(0)))
df_intermediate_hr = df_intermediate_hr.withColumn('visitor_cnt_hour_11', F.when(F.col("par_hour") == 11, F.lit(1)).otherwise(F.lit(0)))
df_intermediate_hr = df_intermediate_hr.withColumn('visitor_cnt_hour_12', F.when(F.col("par_hour") == 12, F.lit(1)).otherwise(F.lit(0)))
df_intermediate_hr = df_intermediate_hr.withColumn('visitor_cnt_hour_13', F.when(F.col("par_hour") == 13, F.lit(1)).otherwise(F.lit(0)))
df_intermediate_hr = df_intermediate_hr.withColumn('visitor_cnt_hour_14', F.when(F.col("par_hour") == 14, F.lit(1)).otherwise(F.lit(0)))
df_intermediate_hr = df_intermediate_hr.withColumn('visitor_cnt_hour_15', F.when(F.col("par_hour") == 15, F.lit(1)).otherwise(F.lit(0)))
df_intermediate_hr = df_intermediate_hr.withColumn('visitor_cnt_hour_16', F.when(F.col("par_hour") == 16, F.lit(1)).otherwise(F.lit(0)))
df_intermediate_hr = df_intermediate_hr.withColumn('visitor_cnt_hour_17', F.when(F.col("par_hour") == 17, F.lit(1)).otherwise(F.lit(0)))
df_intermediate_hr = df_intermediate_hr.withColumn('visitor_cnt_hour_18', F.when(F.col("par_hour") == 18, F.lit(1)).otherwise(F.lit(0)))
df_intermediate_hr = df_intermediate_hr.withColumn('visitor_cnt_hour_19', F.when(F.col("par_hour") == 19, F.lit(1)).otherwise(F.lit(0)))
df_intermediate_hr = df_intermediate_hr.withColumn('visitor_cnt_hour_20', F.when(F.col("par_hour") == 20, F.lit(1)).otherwise(F.lit(0)))
df_intermediate_hr = df_intermediate_hr.withColumn('visitor_cnt_hour_21', F.when(F.col("par_hour") == 21, F.lit(1)).otherwise(F.lit(0)))
df_intermediate_hr = df_intermediate_hr.withColumn('visitor_cnt_hour_22', F.when(F.col("par_hour") == 22, F.lit(1)).otherwise(F.lit(0)))
df_intermediate_hr = df_intermediate_hr.withColumn('visitor_cnt_hour_23', F.when(F.col("par_hour") == 23, F.lit(1)).otherwise(F.lit(0)))

# df_intermediate_hr.display()

In [0]:
# home 
df_intermediate_placetype = df_intermediate_hr.withColumn('home', F.when(F.col("place_type") == 'home', F.lit(1)).otherwise(F.lit(0)))
# work
df_intermediate_placetype = df_intermediate_placetype.withColumn('work', F.when(F.col("place_type") == 'work', F.lit(1)).otherwise(F.lit(0)))
# home_work
df_intermediate_placetype = df_intermediate_placetype.withColumn('home_work', F.when(F.col("place_type") == 'home_work', F.lit(1)).otherwise(F.lit(0)))
# thirdplace
df_intermediate_placetype = df_intermediate_placetype.withColumn('thirdplace', F.when(F.col("place_type") == 'thirdplace', F.lit(1)).otherwise(F.lit(0)))
# home_work_unidentified
df_intermediate_placetype = df_intermediate_placetype.withColumn('home_work_unidentified', F.when(F.col("place_type") == 'home_work_unidentified', F.lit(1)).otherwise(F.lit(0)))

df_intermediate_placetype = df_intermediate_placetype.withColumn('other_placetype', F.when(F.col("place_type") == 'other', F.lit(1)).otherwise(F.lit(0)))

# df_intermediate_placetype.display()

In [0]:
df_intermediate_province = df_intermediate_placetype.withColumn('bangkok', F.when(F.lower(F.col("home_province")) == 'bangkok', F.lit(1)).otherwise(F.lit(0)))
df_intermediate_province = df_intermediate_province.withColumn('nonthaburi', F.when(F.lower(F.col("home_province")) == 'nonthaburi', F.lit(1)).otherwise(F.lit(0)))
df_intermediate_province = df_intermediate_province.withColumn('nakhonpathom', F.when(F.lower(F.col("home_province")) == 'nakhonpathom', F.lit(1)).otherwise(F.lit(0)))
df_intermediate_province = df_intermediate_province.withColumn('pathumthani', F.when(F.lower(F.col("home_province")) == 'pathumthani', F.lit(1)).otherwise(F.lit(0)))
df_intermediate_province = df_intermediate_province.withColumn('samutprakan', F.when(F.lower(F.col("home_province")) == 'samutprakan', F.lit(1)).otherwise(F.lit(0)))
df_intermediate_province = df_intermediate_province.withColumn('samutsakhon', F.when(F.lower(F.col("home_province")) == 'samutsakhon', F.lit(1)).otherwise(F.lit(0)))

# df_intermediate_province.display()

In [0]:
df_intermediate_region = df_intermediate_province.withColumn('central', F.when(F.col("region") == 'central', F.lit(1)).otherwise(F.lit(0)))
df_intermediate_region = df_intermediate_region.withColumn('eastern', F.when(F.col("region") == 'eastern', F.lit(1)).otherwise(F.lit(0)))
df_intermediate_region = df_intermediate_region.withColumn('north', F.when(F.col("region") == 'north', F.lit(1)).otherwise(F.lit(0)))
df_intermediate_region = df_intermediate_region.withColumn('northeastern', F.when(F.col("region") == 'northeastern', F.lit(1)).otherwise(F.lit(0)))
df_intermediate_region = df_intermediate_region.withColumn('south', F.when(F.col("region") == 'south', F.lit(1)).otherwise(F.lit(0)))
df_intermediate_region = df_intermediate_region.withColumn('west', F.when(F.col("region") == 'west', F.lit(1)).otherwise(F.lit(0)))


In [0]:
# Flag nationality_groups
df_intermediate_country = df_intermediate_region.withColumn('middle_east', F.when(F.col("nationality_group") == 'Middle East', F.lit(1)).otherwise(F.lit(0)))
df_intermediate_country = df_intermediate_country.withColumn('africa', F.when(F.col("nationality_group") == 'Africa', F.lit(1)).otherwise(F.lit(0)))
df_intermediate_country = df_intermediate_country.withColumn('europe', F.when(F.col("nationality_group") == 'Europe', F.lit(1)).otherwise(F.lit(0)))
df_intermediate_country = df_intermediate_country.withColumn('asia', F.when(F.col("nationality_group") == 'Asia', F.lit(1)).otherwise(F.lit(0)))
df_intermediate_country = df_intermediate_country.withColumn('oceania', F.when(F.col("nationality_group") == 'Oceania', F.lit(1)).otherwise(F.lit(0)))
df_intermediate_country = df_intermediate_country.withColumn('north_america', F.when(F.col("nationality_group") == 'North America', F.lit(1)).otherwise(F.lit(0)))
df_intermediate_country = df_intermediate_country.withColumn('south_america', F.when(F.col("nationality_group") == 'South America', F.lit(1)).otherwise(F.lit(0)))
df_intermediate_country = df_intermediate_country.withColumn('russian', F.when(F.col("nationality_group") == 'Russian', F.lit(1)).otherwise(F.lit(0)))
df_intermediate_country = df_intermediate_country.withColumn('chinese', F.when(F.col("nationality_group") == 'Chinese', F.lit(1)).otherwise(F.lit(0)))
df_intermediate_country = df_intermediate_country.withColumn('indian', F.when(F.col("nationality_group") == 'Indian', F.lit(1)).otherwise(F.lit(0)))
df_intermediate_country = df_intermediate_country.withColumn('high_spending_asians', F.when(F.col("nationality_group") == 'High Spending Asians', F.lit(1)).otherwise(F.lit(0)))
df_intermediate_country = df_intermediate_country.withColumn('clmv_other_asians', F.when(F.col("nationality_group") == 'CLMV+Other Asians', F.lit(1)).otherwise(F.lit(0)))
df_intermediate_country = df_intermediate_country.withColumn('other', F.when(F.col('demo_tourist_sim_v1_tourist_bin') == 'y', F.lit(1)).otherwise(F.lit(0)))

In [0]:
core_columns = [
    'month',
    'latitude',
    'longitude',
    'mall',
    'province',
    'district',
    'sub_district'
]

In [0]:
df_intermediate_country = df_intermediate_country.withColumnRenamed('name','mall')

In [0]:
df_bmr = df_intermediate_country.filter(F.col('is_bmr')==1)
df_agg_bmr =(df_bmr
    .groupby(core_columns)
    .agg(
        F.count('msisdn').alias('total_mall_hourly_visit_cnt'),
        F.sum('gender_female').alias('gender_female'),
        F.sum('gender_male').alias('gender_male'),
        F.sum('gender_unidentified').alias('gender_unidentified'),
        F.sum('age_1_12').alias('age_1_12'),
        F.sum('age_13_17').alias('age_13_17'),
        F.sum('age_18_24').alias('age_18_24'),
        F.sum('age_25_34').alias('age_25_34'),
        F.sum('age_35_44').alias('age_35_44'),
        F.sum('age_45_54').alias('age_45_54'),
        F.sum('age_55_59').alias('age_55_59'),
        F.sum('age_60_64').alias('age_60_64'),
        F.sum('age>=65').alias('age>=65'),
        F.sum('age_unidentified').alias('age_unidentified'),
        F.sum('holiday_weekend').alias('holiday_weekend'),
        F.sum('normal_weekend').alias('normal_weekend'),
        F.sum('special_weekend').alias('special_weekend'),
        F.sum('normal_weekday').alias('normal_weekday'),
        F.sum('special_weekday').alias('special_weekday'),
        F.sum('visitor_cnt_hour_10').alias('visitor_cnt_hour_10'),
        F.sum('visitor_cnt_hour_11').alias('visitor_cnt_hour_11'),
        F.sum('visitor_cnt_hour_12').alias('visitor_cnt_hour_12'),
        F.sum('visitor_cnt_hour_13').alias('visitor_cnt_hour_13'),
        F.sum('visitor_cnt_hour_14').alias('visitor_cnt_hour_14'),
        F.sum('visitor_cnt_hour_15').alias('visitor_cnt_hour_15'),
        F.sum('visitor_cnt_hour_16').alias('visitor_cnt_hour_16'),
        F.sum('visitor_cnt_hour_17').alias('visitor_cnt_hour_17'),
        F.sum('visitor_cnt_hour_18').alias('visitor_cnt_hour_18'),
        F.sum('visitor_cnt_hour_19').alias('visitor_cnt_hour_19'),
        F.sum('visitor_cnt_hour_20').alias('visitor_cnt_hour_20'),
        F.sum('visitor_cnt_hour_21').alias('visitor_cnt_hour_21'),
        F.sum('visitor_cnt_hour_22').alias('visitor_cnt_hour_22'),
        F.sum('visitor_cnt_hour_23').alias('visitor_cnt_hour_23'),
        F.sum('home').alias('home'),
        F.sum('work').alias('work'),
        F.sum('home_work').alias('home_work'),
        F.sum('thirdplace').alias('thirdplace'),
        F.sum('home_work_unidentified').alias('home_work_unidentified'),
        F.sum('other_placetype').alias('other_placetype'),
        F.sum('bangkok').alias('bangkok'),
        F.sum('nonthaburi').alias('nonthaburi'),
        F.sum('pathumthani').alias('pathumthani'),
        F.sum('samutprakan').alias('samutprakan'),
        F.sum('nakhonpathom').alias('nakhonpathom'),
        F.sum('samutsakhon').alias('samutsakhon')

    )
)

In [0]:
df_non_bmr = df_intermediate_country.filter(F.col('is_non_bmr')==1)
df_agg_non_bmr =(df_non_bmr
    .groupby(core_columns)
    .agg(
        F.count('msisdn').alias('total_mall_hourly_visit_cnt'),
        F.sum('gender_female').alias('gender_female'),
        F.sum('gender_male').alias('gender_male'),
        F.sum('gender_unidentified').alias('gender_unidentified'),
        F.sum('age_1_12').alias('age_1_12'),
        F.sum('age_13_17').alias('age_13_17'),
        F.sum('age_18_24').alias('age_18_24'),
        F.sum('age_25_34').alias('age_25_34'),
        F.sum('age_35_44').alias('age_35_44'),
        F.sum('age_45_54').alias('age_45_54'),
        F.sum('age_55_59').alias('age_55_59'),
        F.sum('age_60_64').alias('age_60_64'),
        F.sum('age>=65').alias('age>=65'),
        F.sum('age_unidentified').alias('age_unidentified'),
        F.sum('holiday_weekend').alias('holiday_weekend'),
        F.sum('normal_weekend').alias('normal_weekend'),
        F.sum('special_weekend').alias('special_weekend'),
        F.sum('normal_weekday').alias('normal_weekday'),
        F.sum('special_weekday').alias('special_weekday'),
        F.sum('visitor_cnt_hour_10').alias('visitor_cnt_hour_10'),
        F.sum('visitor_cnt_hour_11').alias('visitor_cnt_hour_11'),
        F.sum('visitor_cnt_hour_12').alias('visitor_cnt_hour_12'),
        F.sum('visitor_cnt_hour_13').alias('visitor_cnt_hour_13'),
        F.sum('visitor_cnt_hour_14').alias('visitor_cnt_hour_14'),
        F.sum('visitor_cnt_hour_15').alias('visitor_cnt_hour_15'),
        F.sum('visitor_cnt_hour_16').alias('visitor_cnt_hour_16'),
        F.sum('visitor_cnt_hour_17').alias('visitor_cnt_hour_17'),
        F.sum('visitor_cnt_hour_18').alias('visitor_cnt_hour_18'),
        F.sum('visitor_cnt_hour_19').alias('visitor_cnt_hour_19'),
        F.sum('visitor_cnt_hour_20').alias('visitor_cnt_hour_20'),
        F.sum('visitor_cnt_hour_21').alias('visitor_cnt_hour_21'),
        F.sum('visitor_cnt_hour_22').alias('visitor_cnt_hour_22'),
        F.sum('visitor_cnt_hour_23').alias('visitor_cnt_hour_23'),
        F.sum('central').alias('central'),
        F.sum('eastern').alias('eastern'),
        F.sum('northeastern').alias('northeastern'),
        F.sum('west').alias('west'),
        F.sum('north').alias('north'),
        F.sum('south').alias('south'),
    )
)

In [0]:
df_foriegner = df_intermediate_country.filter(F.col('is_foriegner')==1)
df_agg_foriegner =(df_foriegner
    .groupby(core_columns)
    .agg(
        F.count('msisdn').alias('total_mall_hourly_visit_cnt'),
        F.sum('gender_female').alias('gender_female'),
        F.sum('gender_male').alias('gender_male'),
        F.sum('gender_unidentified').alias('gender_unidentified'),
        F.sum('age_1_12').alias('age_1_12'),
        F.sum('age_13_17').alias('age_13_17'),
        F.sum('age_18_24').alias('age_18_24'),
        F.sum('age_25_34').alias('age_25_34'),
        F.sum('age_35_44').alias('age_35_44'),
        F.sum('age_45_54').alias('age_45_54'),
        F.sum('age_55_59').alias('age_55_59'),
        F.sum('age_60_64').alias('age_60_64'),
        F.sum('age>=65').alias('age>=65'),
        F.sum('age_unidentified').alias('age_unidentified'),
        F.sum('holiday_weekend').alias('holiday_weekend'),
        F.sum('normal_weekend').alias('normal_weekend'),
        F.sum('special_weekend').alias('special_weekend'),
        F.sum('normal_weekday').alias('normal_weekday'),
        F.sum('special_weekday').alias('special_weekday'),
        F.sum('visitor_cnt_hour_10').alias('visitor_cnt_hour_10'),
        F.sum('visitor_cnt_hour_11').alias('visitor_cnt_hour_11'),
        F.sum('visitor_cnt_hour_12').alias('visitor_cnt_hour_12'),
        F.sum('visitor_cnt_hour_13').alias('visitor_cnt_hour_13'),
        F.sum('visitor_cnt_hour_14').alias('visitor_cnt_hour_14'),
        F.sum('visitor_cnt_hour_15').alias('visitor_cnt_hour_15'),
        F.sum('visitor_cnt_hour_16').alias('visitor_cnt_hour_16'),
        F.sum('visitor_cnt_hour_17').alias('visitor_cnt_hour_17'),
        F.sum('visitor_cnt_hour_18').alias('visitor_cnt_hour_18'),
        F.sum('visitor_cnt_hour_19').alias('visitor_cnt_hour_19'),
        F.sum('visitor_cnt_hour_20').alias('visitor_cnt_hour_20'),
        F.sum('visitor_cnt_hour_21').alias('visitor_cnt_hour_21'),
        F.sum('visitor_cnt_hour_22').alias('visitor_cnt_hour_22'),
        F.sum('visitor_cnt_hour_23').alias('visitor_cnt_hour_23'),
        F.sum('middle_east').alias('middle_east'),
        F.sum('africa').alias('africa'),
        F.sum('europe').alias('europe'),
        F.sum('asia').alias('asia'),
        F.sum('oceania').alias('oceania'),
        F.sum('north_america').alias('north_america'),
        F.sum('south_america').alias('south_america'),
        F.sum('russian').alias('russian'),
        F.sum('chinese').alias('chinese'),
        F.sum('indian').alias('indian'),
        F.sum('high_spending_asians').alias('high_spending_asians'),
        F.sum('clmv_other_asians').alias('clmv_other_asians'),
        F.sum('other').alias('other')
        
    )
)

In [0]:
save_to_csv(df_agg_bmr, report_path+ f"report4/report4_bmr_{par_month}.csv")
save_to_csv(df_agg_non_bmr, report_path+ f"report4/report4_non_bmr_{par_month}.csv")
save_to_csv(df_agg_foriegner, report_path+ f"report4/report4_foreigner_{par_month}.csv")